## **<font color="blue">Define and Register Plugins</font>**
This section explains how to define Plugin classes and register them as part of your agent workflow.

### **Create Plugin Class**

In [1]:
# count_plugin.py

from google.adk.agents.base_agent import BaseAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models.llm_request import LlmRequest
from google.adk.plugins.base_plugin import BasePlugin


class CountInvocationPlugin(BasePlugin):
  """A custom plugin that counts agent and tool invocations."""

  def __init__(self) -> None:
    """Initialize the plugin with counters."""
    super().__init__(name="count_invocation")
    self.agent_count: int = 0
    self.tool_count: int = 0
    self.llm_request_count: int = 0

  async def before_agent_callback(
      self, *, agent: BaseAgent, callback_context: CallbackContext
  ) -> None:
    """Count agent runs."""
    self.agent_count += 1
    print(f"[Plugin] Agent run count: {self.agent_count}")

  async def before_model_callback(
      self, *, callback_context: CallbackContext, llm_request: LlmRequest
  ) -> None:
    """Count LLM requests."""
    self.llm_request_count += 1
    print(f"[Plugin] LLM request count: {self.llm_request_count}")

### **Register Plugin Class**

In [2]:
# main.py

import asyncio

from google.adk import Agent
from google.adk.runners import InMemoryRunner
from google.adk.tools.tool_context import ToolContext
from google.genai import types

# [Step 2] Import the plugin.
# from .count_plugin import CountInvocationPlugin


async def hello_world(tool_context: ToolContext, query: str):
  print(f'Hello world: query is [{query}]')


root_agent = Agent(
    model='gemini-2.0-flash',
    name='hello_world',
    description='Prints hello world with user query.',
    instruction="""Use hello_world tool to print hello world and user query.
    """,
    tools=[hello_world],
)


async def main():
  """Main entry point for the agent."""
  prompt = 'hello world'
  runner = InMemoryRunner(
      agent=root_agent,
      app_name='test_app_with_plugin',
      # [Step 2] Add your plugin here. You can add multiple plugins.
      plugins=[CountInvocationPlugin()],
  )
  session = await runner.session_service.create_session(
      user_id='user',
      app_name='test_app_with_plugin',
  )

  async for event in runner.run_async(
      user_id='user',
      session_id=session.id,
      new_message=types.Content(
          role='user', parts=[types.Part.from_text(text=prompt)]
      ),
  ):
    print(f'** Got event from {event.author}')


await main()
      
# if __name__ == '__main__':
#   asyncio.run(main())

[Plugin] Agent run count: 1
[Plugin] LLM request count: 1


D:\Agent-Development-Kit\venv\Lib\site-packages\google\adk\flows\llm_flows\base_llm_flow.py:449: UserWarning: [EXPERIMENTAL] feature FeatureName.PROGRESSIVE_SSE_STREAMING is enabled.
  async for event in agen:


** Got event from hello_world
Hello world: query is [hello world]
** Got event from hello_world
[Plugin] LLM request count: 2
** Got event from hello_world


## **<font color="blue">Build Workflows with Plugins</font>**
- A **hook** is just a specific point in a program where you can insert your own code. Think of it as a checkpoint or event in the workflow that “calls out” to let you run some code.
- Plugin callback hooks are a mechanism for implementing logic that intercepts, modifies, and even controls the agent's execution lifecycle. Each hook is a specific method in your Plugin class that you can implement to run code at a key moment. You have a choice between two modes of operation based on your hook's return value:
  - **To Observer: (No Return Value)**
    - Implement a hook with no return value (`None`). This approach is for tasks such as logging or collecting metrics, as it allows the agent's workflow to proceed to the next step without interruption.
    - For example, you could use `after_tool_callback` in a Plugin to log every tool's result for debugging.
    - Technical View:
      - Your hook doesn't return anythong (None).
      - The agent continues normally after your hook.
      - Use cases: Logging, monitoring, collecting metrics.
      - Exmple: `after_tool_callback` logs the output of every tool the agent uses.
        ```python
        def after_tool_callback(self, context, tool_result):
            print("Tool result:", tool_result)  # Just log it
            # No return -> workflow continues normally
        ```
  - **To Intervene: (Return a Value)**
    - Implement a hook and return a value. This approach short-circuits the workflow. The `Runner` halts processing, skip any subsequent plugins and the original intended action, like a Model call, and use a Plugin callback's return value as the result.
    - A common use case is implementing `before_model_callback` to return a cached `LlmResponse`, preventing a redundant and costly API call.
    - Technical View:
      - Your hook returns a value.
      - This stops the normal workflow immediately.
      - Skips any remaining plugins or the intended action (like calling the model).
      - Use case: Return a cached response instead of calling the model again.
        ```python
        def before_model_callback(self, context):
            if cached_result := get_cached_response(context.prompt):
                return cached_result  # Stops the model call, uses cached response
        ```
  - **To Amend: (Modify Context)**
    - Implement a hook and modify the Context object. This approach allows you to modify the context data for the module to be executed without otherwise interrupting the execution of that module.
    - For example, adding additional, standardized prompt text for Model object execution.
    - Technical View:
      - Your hook changes the context object.
      - Workflow continues, but the modules sees the modified context.
      - User case: Add extra instructions to a prompt before sending to a model.
        ```python
        def before_model_callback(self, context):
            context.prompt += "\nAlways answer in bullet points."  # Modify the prompt
            # No return -> workflow continues
        ```
- **Caution:**
  - Plugin callback funcitons have precedence over callbacks implemented at the object level. This behavior means that Any Plugin callbacks code is executed _before_ any Agent, Model, or Tool objects callbacks are executed. Futhermore, if a Plugin-level agent callback return any value, and not any empty(`None`) response, the Agent, Mode, or Tool-level callback is _not executed_ (skipped).
  - The Plugin desing establishes a hierarchy of code execution and separates global concerns from local agent logic. A Plugin is the stateful _module_ you build, such as `PerformanceMonitoringPlugin`, while the callback hooks are the specific _functions_ within that module that get executed. This architecture differs fundamentally from standard Agent Callbacks in these critical ways:
    - **Scope:** Plugin hooks are _global_. You register a Plugin once on the `Runner`, and its hooks apply universally to every Agent, Model, and Tool it manages. In contrast, Agent Callbacks are _local_, configured individually on a specific agent instance.
    - **Execution Order:** Plugins have _precedence_. For any given event, the Plugin hooks always run before any corresponding Agent Callback. This system behavior makes Plugins the correct architectural choice for implementing cross-cutting features like security policies, universal caching, and consistent logging across your entire application.


| Feature                  | Plugin Hooks (Global)                               | Agent Callbacks (Local)                     |
|---------------------------|---------------------------------------------------|--------------------------------------------|
| **Scope**                 | Global – applies to all Agents, Models, Tools   | Local – only applies to a specific Agent  |
| **Execution Order**       | Runs first, before any Agent/Model/Tool callbacks| Runs after Plugin hooks                     |
| **Return Value Behavior** | Returning a value **skips object-level callbacks**; returning `None` continues workflow | Always executed if workflow reaches it      |
| **Purpose**               | Cross-cutting concerns: logging, caching, security, monitoring | Custom agent-specific behavior             |
| **Stateful Module**       | Yes – can maintain metrics, cache, state        | Usually stateless                           |
| **Examples**              | `PerformanceMonitoringPlugin`, `SecurityPolicyPlugin` | Custom `before_call` or `after_call` on one agent |
| **Modes**                 | Observe (log), Intervene (short-circuit), Amend (modify context) | Mostly observe or amend agent behavior     |

### **Agent Callbacks and Plugins**
As mentioned the previous section, there are some functional similarities between Plugins and Agent Callbacks.
|                           | Plugins                                           | Agent Callbacks                              |
|---------------------------|---------------------------------------------------|--------------------------------------------|
| **Scope**                 | **Global:** Apply to all _agents_ / _tools_ / _LLMs_ in the `Runner`   | **Local:** Apply only to the specific agent instance they are configured on.  |
| **Primary Use Case**       | **Horizonal Features:** Logging, policy, monitoring, global caching. | **Specific Agent Logic:** Modifying the behavior or state of the single agent.|
| **Configuration** | Configure once on the `Runner` | Configure individually on each `BaseAgent` instance. |
| **Execution Order** | Plugin callbacks run __before__ Agent Callbacks. | Agent callbacks run **after** Plugin callbacks. |

## **<font color="blue">Plugin Callback Hooks</font>**
- You define when a Plugin is called with the callback functions to define in your Plugin class. Callbacks are available when a user message is received, before and after an `Runner`, `Agent`, `Model`, or `Tool` is called, for `Events`, and  when a `Model`, or `Tool` error occurs. These callbacks include, and take precedence over, the any callbacks defined within your Agent, Model, and Tool classes.

### **<font color="green">User Message Callbacks</font>**
- A _User Message_ callback (`on_user_message_callback`) happens when a user sends a message. The `on_user_message_callback` is the very first hook to run, giving you a chance to inspect or modify the initial input.
  - **When It Runs:** This callback happens immediately after `runner.run()`, before any other processing.
  - **Purpose:** The first opportunity to inspect or modify the user's raw input.
  - **Flow Control:** Returns a `types.Content` object to __replace__ the user's original message.
```python
async def on_user_message_callback(
    self,
    *,
    invocation_context: InvocationContext,
    user_message: types.Content,
) -> Optional[types.Content]:
```

### **<font color="green">Runner Start Callbacks</font>**
- A _Runner start_ callback (`before_run_callback`) happens when the `Runner` object takes the potentially modified user message and prepares for execution. The `before_run_callback` fires here, allowing for global setup before any agent logic begins.
  - **When It Runs:** Immediately after `runner.run()` is called, before any other processing.
  - **Purpose:** The first opportunity to inspect or modify the user's raw input.
  - **Flow Control:** Return a `types.Content` object to **replace** the user's original message.
```python
async def before_run_callback(
    self, *, invocation_context: InvocationContext
) -> Optional[types.Content]:
```

### **<font color="green">Agent Execution Callbacks</font>**
- _Agent execution_ callbacks (`before_agent`, `after_agent`) happen when a `Runner` object invokes and agent. The `before_agent_callback` runs immediately before the agent's main work begins.
- The main work encompasses the agent's entire process for handling the request, which could involve calling models or tools. After the agent has finished all its steps and prepared a result, the `after_agent_callback` runs.
- **Caution:** Plugin that implement these callbacks are executed _before_ the Agent-level callbacks are executed. Furthermore, if a Plugin-level agent callback return anything other than a `None` or null response, the Agent-level callback is _not executed_ (skipped).

### **<font color="green">Model Callbacks</font>**
- Model callbacks (**`before_model`, `after_model`, `on_model_error`**) happen before and after a Model object executes. The Plugins feature also supports a callback in the event of an error, as detailed below:
  - If an agent needs to call an AI model, `before_model_callback` runs first.
  - If the model call is successful, `after_model_callback` runs next.
  - If the model call fails with an exception, the `on_model_error_callback` is triggered instead, allowing for graceful recovery.
- **Caution:** Plugins that implement the `before_model` and `after_model` _callback methods are executed before_ the Model-level callbacks are executed. Furthermore, if a Plugin-level model callback returns anything other than a `None` or null response, the Model-level callback is _not executed_ (skipped).
- **Model on error callback details:** The on error callback for Model objects is only supported by the Plugins feature works as follows:
  - **When It Runs:** When an execution is raised during the model call.
  - **Common Use Cases:** Graceful error handling, logging the specific error, or returning a fallback response, such as, "The AI service is currently unavailable."
  - **Flow Control:**
    - Returns an `LlmResponse` object to support the exception and provide a fallback result.
    - Returns `None` to allow the original exception to be raised.
- **Note:** If the execution of the Model object returns a `LlmResponse`, the system resumes the execution flow, and `after_model_callback` will be triggered noramlly.
```python
async def on_model_error_callback(
    self,
    *,
    callback_context: CallbackContext,
    llm_request: LlmRequest,
    error: Exception,
) -> Optional[LlmResponse]:
```

### **<font color="green">Tool Callbacks</font>**
- Tool callbacks (**`before_tool`, `after_tool`, `on_tool_error`**) for Plugins happen before or after the execution of a tool, or when an error occurs. The Plugins feature also supports a callback in the event of an error, as detailed below:
  - When an agent executes a Tool, `before_tool_callback` runs first.
  - If the tool executes successfully, `after_tool_callback` runs next.
  - If the tool raises an exception, the `on_tool_error_callback` is triggered instead, giving you a chance to handle the failure. If `on_tool_error_callback` returns a dict, `after_tool_callback` will be triggered normally.
- **Caution:** Plugins that implement these callbacks are executed before the Tool-level callbacks are executed. Furthermore, if a Plugin-level tool callback returns anything other than a `None` or null response, the Tool-level callback is not executed (skipped).
- **Tool on error callback details:** The on error callback for Tool objects is only supported by the Plugins feature works as follows:
  - **When It Runs:** When an execution is raised during the execution of a tool's `run` method.
  - **Purpose:** Catching specific tool exception (like `APIError`), logging the failure, and providing a user-friendly error message back to the LLM.
  - **Flow Control:** Return a `dict` to suppress the exception, provide a fallback result. Return `None` to allow the original exception to be raised.
- **Note:** By returning a `dict`, this resumes the execution flow, and `after_tool_callback` will be triggered noramally.
```python
async def on_tool_error_callback(
    self,
    *,
    tool: BaseTool,
    tool_args: dict[str, Any],
    tool_context: ToolContext,
    error: Exception,
) -> Optional[dict]:
```

### **<font color="green">Event Callbacks</font>**
- An _Event callback_ (`on_event_callback`) happens when an agent produces outputs such as a text response or a tool call result, it yields them as `Event` objects. The `on_event_callback` fires for each event, allowing you to modify it before it's streamed to the client.
  - **When It Runs:** After an agent yields an `Event` but before it's sent to the user. An agent's run may produce multiple events.
  - **Purpose:** Useful for modifying or enriching events (e.g., adding metadata) or for triggering side effects based on specific events.
  - **Flow Control:** Return an `Event` object to replace the original event.
```python
async def on_event_callback(
    self, *, invocation_context: InvocationContext, event: Event
) -> Optional[Event]:
```

### **<font color="green">Runner End Callbacks</font>**
- The _Runner end_ callback (**`after_run_callback`**) happens when the agent has finished its entire process and all events has been handled, the `Runner` completes its run.
- The `after_run_callback` is the final hook, perfect for cleanup and final reporting.
  - **When It Runs:** After the `Runner` fully completes the execution of a request.
  - **Purpose:** Ideal for global cleanup tasks, such as closing connections or finalizing logs and metrics data.
  - **Flow Control:** This callback is for teardown only and cannot alter the final result.
```python
async def after_run_callback(
    self, *, invocation_context: InvocationContext
) -> Optional[None]:
```


In [5]:
# -----------------------------
# chatbot_plugin.py
# -----------------------------
import os
import nest_asyncio
from typing import Optional, Any, Dict
import google.genai.types as types

from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.plugins import BasePlugin
from config import config

# -----------------------------
# Patch asyncio for JupyterLab
# -----------------------------
nest_asyncio.apply()

# -----------------------------
# Setup API Key and model
# -----------------------------
APP_NAME = "chatbot_plugin_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Define Plugin with all callback hooks
# -----------------------------
class DemoPlugin(BasePlugin):
    async def on_user_message_callback(
        self, *, invocation_context: Any, user_message: types.Content
    ) -> Optional[types.Content]:
        original_text = user_message.parts[0].text
        print(f"[Plugin] on_user_message_callback: {original_text}")
        user_message.parts[0].text = f"(Processed User) {original_text}"
        return user_message

    async def before_run_callback(self, *, invocation_context: Any) -> Optional[types.Content]:
        print("[Plugin] before_run_callback: Runner preparing")
        return None

    async def before_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        print(f"[Plugin] before_agent_callback: Agent {agent.name} starting")
        return None

    async def after_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        print(f"[Plugin] after_agent_callback: Agent {agent.name} finished")
        return None

    async def before_model_callback(self, *, callback_context: Any, llm_request: Any) -> Optional[Any]:
        print(f"[Plugin] before_model_callback: LLM request={llm_request}")
        return None

    async def after_model_callback(self, *, callback_context: Any, llm_response: Any) -> Optional[Any]:
        print(f"[Plugin] after_model_callback: LLM response={llm_response}")
        return None

    async def on_model_error_callback(
        self, *, callback_context: Any, llm_request: Any, error: Exception
    ) -> Optional[Any]:
        print(f"[Plugin] on_model_error_callback: {error}")
        return types.Content(
            role="assistant",
            parts=[types.Part(text="Sorry, the AI service failed. Please try again later.")]
        )

    async def before_tool_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any
    ) -> Optional[Any]:
        print(f"[Plugin] before_tool_callback: {getattr(tool, 'name', 'tool')}")
        return None

    async def after_tool_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, result: Any
    ) -> Optional[Any]:
        print(f"[Plugin] after_tool_callback: {getattr(tool, 'name', 'tool')}, result={result}")
        return None

    async def on_tool_error_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, error: Exception
    ) -> Optional[Dict[str, Any]]:
        print(f"[Plugin] on_tool_error_callback: {getattr(tool, 'name', 'tool')}, error={error}")
        return {"error": str(error)}

    async def on_event_callback(self, *, invocation_context: Any, event: Any) -> Optional[Any]:
        if hasattr(event, "content") and event.content:
            event.content.parts[0].text = f"(Processed Event) {event.content.parts[0].text}"
        print("[Plugin] on_event_callback:", getattr(event, "type", "event"))
        return event

    async def after_run_callback(self, *, invocation_context: Any) -> Optional[Any]:
        print("[Plugin] after_run_callback: Run complete")
        return None

# -----------------------------
# Create LlmAgent
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="You are a helpful assistant. Respond politely and concisely to user questions."
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# -----------------------------
# Create session ONCE
# -----------------------------
try:
    await session_service.create_session(
        app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
    )
except Exception:
    pass  # Ignore if session already exists

# -----------------------------
# Setup Runner and attach Plugin
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[DemoPlugin(name="DemoPlugin")]
)

# -----------------------------
# Chat Function
# -----------------------------
async def chat(user_input: str):
    """Send a message to the agent and print final responses."""
    content = types.Content(role="user", parts=[types.Part(text=user_input)])
    
    events = runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content
    )

    async for event in events:
        if event.is_final_response() and event.content and event.content.parts:
            print("ChatAgent:", event.content.parts[0].text)

# -----------------------------
# Demo Execution
# -----------------------------
await chat("Hello! Can you suggest a fun weekend activity?")
await chat("Give me a short summary of Artificial Intelligence.")

[Plugin] on_user_message_callback: Hello! Can you suggest a fun weekend activity?
[Plugin] before_run_callback: Runner preparing
[Plugin] before_agent_callback: Agent ChatAgent starting
[Plugin] before_model_callback: LLM request=model='gemini-2.5-flash' contents=[Content(
  parts=[
    Part(
      text='(Processed User) Hello! Can you suggest a fun weekend activity?'
    ),
  ],
  role='user'
)] config=GenerateContentConfig(
  system_instruction="""You are a helpful assistant. Respond politely and concisely to user questions.

You are an agent. Your internal name is "ChatAgent"."""
) live_connect_config=LiveConnectConfig(
  input_audio_transcription=AudioTranscriptionConfig(),
  output_audio_transcription=AudioTranscriptionConfig()
) tools_dict={} cache_config=None cache_metadata=None cacheable_contents_token_count=None previous_interaction_id=None
[Plugin] after_model_callback: LLM response=model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Hello! How a

In [6]:
# -----------------------------
# User Message -> on_user_message_callback -> before_run_callback -> before_agent_callback 
# -> before_model_callback -> Model executes -> after_model_callback -> on_event_callback 
# -> after_agent_callback -> after_run_callback
# -----------------------------
import os
import nest_asyncio
from typing import Optional, Any, Dict
import google.genai.types as types

from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.plugins import BasePlugin
from config import config

# -----------------------------
# Patch asyncio for JupyterLab
# -----------------------------
# Jupyter runs its own event loop, so we patch it to allow nested async calls
nest_asyncio.apply()

# -----------------------------
# Setup API Key and Model
# -----------------------------
APP_NAME = "chatbot_plugin_demo"
USER_ID = "user_001"
SESSION_ID = "session_001"
MODEL = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Define Plugin with all callback hooks
# -----------------------------
class DemoPlugin(BasePlugin):
    """
    Example plugin demonstrating all callback hooks for ADK 1.22.0.
    Logs, modifies messages/events, and handles errors gracefully.
    """

    # -----------------------------
    # User Message Callback
    # -----------------------------
    async def on_user_message_callback(
        self, *, invocation_context: Any, user_message: types.Content
    ) -> Optional[types.Content]:
        """Runs first when a user sends a message."""
        original_text = user_message.parts[0].text
        print(f"[Plugin] on_user_message_callback: {original_text}")
        user_message.parts[0].text = f"(Processed User) {original_text}"
        return user_message

    # -----------------------------
    # Runner Start Callback
    # -----------------------------
    async def before_run_callback(self, *, invocation_context: Any) -> Optional[types.Content]:
        """Runs before Runner starts processing the message."""
        print("[Plugin] before_run_callback: Runner preparing")
        return None

    # -----------------------------
    # Agent Execution Callbacks
    # -----------------------------
    async def before_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        """Runs before the agent processes the message."""
        print(f"[Plugin] before_agent_callback: Agent {agent.name} starting")
        return None

    async def after_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        """Runs after the agent finishes processing the message."""
        print(f"[Plugin] after_agent_callback: Agent {agent.name} finished")
        return None

    # -----------------------------
    # Model Callbacks
    # -----------------------------
    async def before_model_callback(self, *, callback_context: Any, llm_request: Any) -> Optional[Any]:
        """Runs before the model is called."""
        print(f"[Plugin] before_model_callback: LLM request={llm_request}")
        return None

    async def after_model_callback(self, *, callback_context: Any, llm_response: Any) -> Optional[Any]:
        """Runs after the model call finishes."""
        print(f"[Plugin] after_model_callback: LLM response={llm_response}")
        return None

    async def on_model_error_callback(
        self, *, callback_context: Any, llm_request: Any, error: Exception
    ) -> Optional[Any]:
        """Handles errors from model calls gracefully."""
        print(f"[Plugin] on_model_error_callback: {error}")
        return types.Content(
            role="assistant",
            parts=[types.Part(text="Sorry, the AI service failed. Please try again later.")]
        )

    # -----------------------------
    # Tool Callbacks
    # -----------------------------
    async def before_tool_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any
    ) -> Optional[Any]:
        """Runs before a tool executes."""
        print(f"[Plugin] before_tool_callback: {getattr(tool, 'name', 'tool')}")
        return None

    async def after_tool_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, result: Any
    ) -> Optional[Any]:
        """Runs after a tool executes."""
        print(f"[Plugin] after_tool_callback: {getattr(tool, 'name', 'tool')}, result={result}")
        return None

    async def on_tool_error_callback(
        self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, error: Exception
    ) -> Optional[Dict[str, Any]]:
        """Handles tool errors gracefully."""
        print(f"[Plugin] on_tool_error_callback: {getattr(tool, 'name', 'tool')}, error={error}")
        return {"error": str(error)}

    # -----------------------------
    # Event Callback
    # -----------------------------
    async def on_event_callback(self, *, invocation_context: Any, event: Any) -> Optional[Any]:
        """
        Modify events before sending them to the user.
        Prevent duplicate '(Processed Event)' if already applied.
        """
        if hasattr(event, "content") and event.content:
            text = event.content.parts[0].text
            if not text.startswith("(Processed Event)"):
                event.content.parts[0].text = f"(Processed Event) {text}"
        print("[Plugin] on_event_callback:", getattr(event, "type", "event"))
        return event

    # -----------------------------
    # Runner End Callback
    # -----------------------------
    async def after_run_callback(self, *, invocation_context: Any) -> Optional[Any]:
        """Runs after Runner finishes processing all events."""
        print("[Plugin] after_run_callback: Run complete")
        return None

# -----------------------------
# Create LlmAgent (Chatbot)
# -----------------------------
root_agent = LlmAgent(
    model=MODEL,
    name="ChatAgent",
    instruction="You are a helpful assistant. Respond politely and concisely to user questions."
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# -----------------------------
# Setup Runner and Attach Plugin
# -----------------------------
runner = Runner(
    agent=root_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[DemoPlugin(name="DemoPlugin")]  # Attach plugin here
)

# -----------------------------
# Chat Function
# -----------------------------
async def chat(user_input: str):
    """
    Send a message to the agent and print final responses.
    Ensures session is created only once.
    """
    # Create session only if it does not exist
    try:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
    except Exception as e:
        # Session already exists; ignore
        pass

    # Wrap user input in types.Content
    content = types.Content(role="user", parts=[types.Part(text=user_input)])
    
    # Run the agent asynchronously
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    async for event in events:
        # Print final response text
        if event.is_final_response() and event.content and event.content.parts:
            print("ChatAgent:", event.content.parts[0].text)

# -----------------------------
# Run Demo in JupyterLab
# -----------------------------
await chat("Hello! Can you suggest a fun weekend activity?")
await chat("Give me a short summary of Artificial Intelligence.")

[Plugin] on_user_message_callback: Hello! Can you suggest a fun weekend activity?
[Plugin] before_run_callback: Runner preparing
[Plugin] before_agent_callback: Agent ChatAgent starting
[Plugin] before_model_callback: LLM request=model='gemini-2.5-flash' contents=[Content(
  parts=[
    Part(
      text='(Processed User) Hello! Can you suggest a fun weekend activity?'
    ),
  ],
  role='user'
)] config=GenerateContentConfig(
  system_instruction="""You are a helpful assistant. Respond politely and concisely to user questions.

You are an agent. Your internal name is "ChatAgent"."""
) live_connect_config=LiveConnectConfig(
  input_audio_transcription=AudioTranscriptionConfig(),
  output_audio_transcription=AudioTranscriptionConfig()
) tools_dict={} cache_config=None cache_metadata=None cacheable_contents_token_count=None previous_interaction_id=None
[Plugin] after_model_callback: LLM response=model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text='Hello! How a

In [9]:
# -----------------------------
# travel_chatbot_plugin_demo_jupyter.py
# -----------------------------
import os
import nest_asyncio
from typing import Optional, Any, Dict
import google.genai.types as types

from google.adk.sessions import InMemorySessionService
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.agents import LlmAgent
from google.adk.plugins import BasePlugin
from config import config

# -----------------------------
# Patch asyncio for Jupyter
# -----------------------------
# Jupyter runs its own event loop; patch to allow nested async calls
nest_asyncio.apply()

# -----------------------------
# Setup API Key and Model
# -----------------------------
APP_NAME = "travel_chatbot_demo"
USER_ID = "user_001"
SESSION_ID = "session_002"
MODEL = "gemini-2.5-flash"
os.environ["GOOGLE_API_KEY"] = config.GOOGLE_API_KEY.strip()

# -----------------------------
# Define Travel Plugin with All Callbacks
# -----------------------------
class TravelPlugin(BasePlugin):
    """
    Travel plugin demo showing all plugin callback hooks in ADK 1.22.0.
    Logs activity, modifies user messages, events, and handles errors gracefully.
    """

    # -----------------------------
    # User Message Callback
    # -----------------------------
    async def on_user_message_callback(
        self, *, invocation_context: Any, user_message: types.Content
    ) -> Optional[types.Content]:
        original_text = user_message.parts[0].text
        print(f"[Plugin] on_user_message_callback: {original_text}")
        user_message.parts[0].text = f"(Processed User) {original_text}"
        return user_message

    # -----------------------------
    # Runner Start Callback
    # -----------------------------
    async def before_run_callback(self, *, invocation_context: Any) -> Optional[types.Content]:
        print("[Plugin] before_run_callback: Runner preparing")
        return None

    # -----------------------------
    # Agent Callbacks
    # -----------------------------
    async def before_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        print(f"[Plugin] before_agent_callback: Agent {agent.name} starting")
        return None

    async def after_agent_callback(self, *, agent: LlmAgent, callback_context: Any) -> Optional[Any]:
        print(f"[Plugin] after_agent_callback: Agent {agent.name} finished")
        return None

    # -----------------------------
    # Model Callbacks
    # -----------------------------
    async def before_model_callback(self, *, callback_context: Any, llm_request: Any) -> Optional[Any]:
        print(f"[Plugin] before_model_callback: LLM request={llm_request}")
        return None

    async def after_model_callback(self, *, callback_context: Any, llm_response: Any) -> Optional[Any]:
        print(f"[Plugin] after_model_callback: LLM response={llm_response}")
        return None

    async def on_model_error_callback(
        self, *, callback_context: Any, llm_request: Any, error: Exception
    ) -> Optional[Any]:
        print(f"[Plugin] on_model_error_callback: {error}")
        return types.Content(
            role="assistant",
            parts=[types.Part(text="Sorry, the AI service failed. Please try again later.")]
        )

    # -----------------------------
    # Tool Callbacks (Demo placeholder)
    # -----------------------------
    async def before_tool_callback(self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any) -> Optional[Any]:
        print(f"[Plugin] before_tool_callback: {getattr(tool, 'name', 'tool')}")
        return None

    async def after_tool_callback(self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, result: Any) -> Optional[Any]:
        print(f"[Plugin] after_tool_callback: {getattr(tool, 'name', 'tool')}, result={result}")
        return None

    async def on_tool_error_callback(self, *, tool: Any, tool_args: Dict[str, Any], tool_context: Any, error: Exception) -> Optional[Dict[str, Any]]:
        print(f"[Plugin] on_tool_error_callback: {getattr(tool, 'name', 'tool')}, error={error}")
        return {"error": str(error)}

    # -----------------------------
    # Event Callback
    # -----------------------------
    async def on_event_callback(self, *, invocation_context: Any, event: Any) -> Optional[Any]:
        if hasattr(event, "content") and event.content:
            # Prevent multiple "(Processed Event)" prefix
            text = event.content.parts[0].text
            if not text.startswith("(Processed Event)"):
                event.content.parts[0].text = f"(Processed Event) {text}"
        print("[Plugin] on_event_callback:", getattr(event, "type", "event"))
        return event

    # -----------------------------
    # Runner End Callback
    # -----------------------------
    async def after_run_callback(self, *, invocation_context: Any) -> Optional[Any]:
        print("[Plugin] after_run_callback: Run complete")
        return None

# -----------------------------
# Create Agent
# -----------------------------
travel_agent = LlmAgent(
    model=MODEL,
    name="TravelAgent",
    instruction="You are a helpful travel assistant. Provide concise, friendly, and practical travel advice."
)

# -----------------------------
# Setup In-Memory Services
# -----------------------------
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# -----------------------------
# Setup Runner with Plugin
# -----------------------------
runner = Runner(
    agent=travel_agent,
    app_name=APP_NAME,
    session_service=session_service,
    artifact_service=artifact_service,
    plugins=[TravelPlugin(name="TravelPlugin")]  # Attach plugin
)

# -----------------------------
# Chat Function (Fixed Session Handling)
# -----------------------------
async def chat(user_input: str):
    """Send message to agent and print verbose final responses."""

    # -----------------------------
    # Ensure session exists (avoid AlreadyExistsError)
    # -----------------------------
    try:
        await session_service.create_session(
            app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID
        )
    except Exception as e:
        # Ignore if session already exists
        if "already exists" in str(e).lower():
            pass
        else:
            raise e

    # -----------------------------
    # Wrap user input
    # -----------------------------
    content = types.Content(role="user", parts=[types.Part(text=user_input)])

    # -----------------------------
    # Run agent and stream events
    # -----------------------------
    events = runner.run_async(user_id=USER_ID, session_id=SESSION_ID, new_message=content)

    async for event in events:
        # Print final response
        if event.is_final_response() and event.content and event.content.parts:
            print("TravelAgent:", event.content.parts[0].text)

# -----------------------------
# Demo Execution (Jupyter)
# -----------------------------
await chat("Hello! Can you suggest a weekend trip destination?")
await chat("Give me a short itinerary for a 2-day trip to Paris.")
await chat("Recommend a few fun activities for families in New York City.")

[Plugin] on_user_message_callback: Hello! Can you suggest a weekend trip destination?
[Plugin] before_run_callback: Runner preparing
[Plugin] before_agent_callback: Agent TravelAgent starting
[Plugin] before_model_callback: LLM request=model='gemini-2.5-flash' contents=[Content(
  parts=[
    Part(
      text='(Processed User) Hello! Can you suggest a weekend trip destination?'
    ),
  ],
  role='user'
)] config=GenerateContentConfig(
  system_instruction="""You are a helpful travel assistant. Provide concise, friendly, and practical travel advice.

You are an agent. Your internal name is "TravelAgent"."""
) live_connect_config=LiveConnectConfig(
  input_audio_transcription=AudioTranscriptionConfig(),
  output_audio_transcription=AudioTranscriptionConfig()
) tools_dict={} cache_config=None cache_metadata=None cacheable_contents_token_count=None previous_interaction_id=None
[Plugin] after_model_callback: LLM response=model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(